## Spark setup part

In [1]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"]             = r"C:\Program Files\Java\jdk-17"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

JDBC_JAR = os.path.join(os.getcwd(), "mssql-jdbc-13.4.0.jre11.jar")

spark = (
    SparkSession.builder
    .appName("AirQualityAnalysis")
    .master("local[*]")
    .config("spark.jars",                    JDBC_JAR)
    .config("spark.driver.extraClassPath",   JDBC_JAR)
    .config("spark.executor.extraClassPath", JDBC_JAR)
    .config("spark.driver.memory",           "2g")
    .config("spark.sql.shuffle.partitions",  "4")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark ready:", spark.version)

✅ Spark ready: 4.1.1


This code starts Spark in local mode and loads the SQL Server JDBC driver.
After this cell runs successfully, you can use `spark` to read CSV, clean data,
do aggregation, and save results to SQL Server.

## Step 2: Read CSV once for testing

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CSV_PATH = "live_input/air_quality_combined.csv"

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "UTF-8")
    .option("nullValue", "")
    .csv(CSV_PATH)
)

print("✅ Raw rows:", raw_df.count())

✅ Raw rows: 4664


## Data Cleaning and Feature Engineering

In [3]:
def clean(df):
    # Step 1 — cast numerics
    for col in ["aqi", "pm25", "pm10", "co", "no2", "so2", "o3"]:
        df = df.withColumn(col, F.col(col).cast(DoubleType()))
    df = df.withColumn("latitude",  F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    # Step 2 — parse datetime
    df = df.withColumn(
        "datetime",
        F.to_timestamp("datetime", "yyyy-MM-dd'T'HH:mm")
    )

    # Step 3 — remove nulls
    df = df.filter(F.col("pm25").isNotNull())
    df = df.filter(F.col("datetime").isNotNull())

    # Step 4 — remove negatives
    df = df.filter(
        (F.col("pm25") >= 0) &
        (F.col("pm10") >= 0) &
        (F.col("aqi")  >= 0)
    )

    # Step 5 — add time columns
    df = df.withColumn("date",  F.to_date("datetime"))
    df = df.withColumn("hour",  F.hour("datetime"))
    df = df.withColumn("month", F.month("datetime"))

    # Step 6 — add category
    df = df.withColumn(
        "air_quality_category",
        F.when(F.col("pm25") <= 12.0,  "Good")
         .when(F.col("pm25") <= 35.4,  "Moderate")
         .when(F.col("pm25") <= 55.4,  "Unhealthy for Sensitive Groups")
         .when(F.col("pm25") <= 150.4, "Unhealthy")
         .when(F.col("pm25") <= 250.4, "Very Unhealthy")
         .otherwise("Hazardous")
    )


    return df

clean_df = clean(raw_df)
print("Raw rows:  ", raw_df.count())
print("Clean rows:", clean_df.count())


Raw rows:   4664
Clean rows: 4664


## Step 4: Data Aggregation and Analysis

In [5]:
avg_pm25 = (
    clean_df.groupBy("city")
    .agg(
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.max("pm25"), 2).alias("max_pm25"),
        F.count("*").alias("total_records")
    )
    .orderBy(F.desc("avg_pm25"))
)

pollutants = (
    clean_df.groupBy("city")
    .agg(
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("co"),   2).alias("avg_co"),
        F.round(F.avg("no2"),  2).alias("avg_no2"),
        F.round(F.avg("so2"),  2).alias("avg_so2"),
        F.round(F.avg("o3"),   2).alias("avg_o3"),
    )
    .orderBy(F.desc("avg_pm25"))
)

dangerous = (
    clean_df.filter(
        F.col("air_quality_category").isin(
            "Unhealthy", "Very Unhealthy", "Hazardous"
        )
    )
    .groupBy("city", "air_quality_category")
    .agg(F.count("*").alias("dangerous_count"))
    .orderBy(F.desc("dangerous_count"))
)

print("✅ Aggregations ready")

✅ Aggregations ready


## Step 5: Connect and Save Results to SQL Server

In [6]:
import urllib
from sqlalchemy import create_engine

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=(local)\\MSSQLSERVER01;"
    "DATABASE=AirQualityDB;"
    "UID=airuser;"
    "PWD=Air@12345;"
    "TrustServerCertificate=yes;"
)

params = urllib.parse.quote_plus(conn_str)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

def save_to_sql(spark_df, table_name, if_exists="append"):
    print(f"\n  Saving → {table_name}")
    try:
        pdf = spark_df.toPandas()
        print(f"    Rows: {len(pdf)}")

        tbl = table_name.replace("dbo.", "")

        pdf.to_sql(
            tbl,
            engine,
            schema="dbo",
            if_exists=if_exists,
            index=False,
            chunksize=100,
            method=None
        )

        print(f"    ✅ Done → {table_name}")

    except Exception:
        import traceback
        print(f"    ❌ FAILED → {table_name}")
        traceback.print_exc()

## Step 7: Batch Processing Loop for Live Updates

In [ ]:
import time
from datetime import datetime
from pyspark.sql import functions as F


CSV_PATH = "live_input/air_quality_combined.csv"
INTERVAL = 60

print("=" * 60)
print("Batch loop started — Interrupt kernel to stop")
print("=" * 60)

batch = 0

while True:
    batch += 1
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n[Batch {batch}] {now}")

    try:
        # 1. Read latest CSV data
        spark.catalog.clearCache()

        raw_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("nullValue", "")
            .option("encoding", "UTF-8")
            .csv(CSV_PATH)
        )

        # 2. Clean data
        clean_df = clean(raw_df)

        raw_count = raw_df.count()
        clean_count = clean_df.count()

        print(f"  Raw rows:   {raw_count}")
        print(f"  Clean rows: {clean_count}")

        # 3. Analysis 1: PM2.5 summary by city
        avg_pm25 = (
            clean_df.groupBy("city")
            .agg(
                F.round(F.avg("pm25"), 2).alias("avg_pm25"),
                F.round(F.max("pm25"), 2).alias("max_pm25"),
                F.count("*").alias("total_records")
            )
            .orderBy(F.desc("avg_pm25"))
        )

        # 4. Analysis 2: Average pollutants by city
        pollutants = (
            clean_df.groupBy("city")
            .agg(
                F.round(F.avg("pm25"), 2).alias("avg_pm25"),
                F.round(F.avg("pm10"), 2).alias("avg_pm10"),
                F.round(F.avg("co"), 2).alias("avg_co"),
                F.round(F.avg("no2"), 2).alias("avg_no2"),
                F.round(F.avg("so2"), 2).alias("avg_so2"),
                F.round(F.avg("o3"), 2).alias("avg_o3")
            )
            .orderBy(F.desc("avg_pm25"))
        )

        # 5. Analysis 3: Dangerous air quality count
        dangerous = (
            clean_df.filter(
                F.col("air_quality_category").isin(
                    "Unhealthy",
                    "Very Unhealthy",
                    "Hazardous"
                )
            )
            .groupBy("city", "air_quality_category")
            .agg(F.count("*").alias("dangerous_count"))
            .orderBy(F.desc("dangerous_count"))
        )

        # 6. Save results to SQL Server
        # Use replace because each batch reads the whole CSV again.
        save_to_sql(clean_df,   "dbo.CleanedAirQuality",  if_exists="replace")
        save_to_sql(avg_pm25,   "dbo.CitySummary",        if_exists="replace")
        save_to_sql(pollutants, "dbo.PollutantAvg",       if_exists="replace")
        save_to_sql(dangerous,  "dbo.DangerousAirQuality", if_exists="replace")

        print(f"  ✅ Batch {batch} completed successfully")

    except Exception:
        import traceback
        print(f"  ❌ Batch {batch} failed")
        traceback.print_exc()

    print(f"  Waiting {INTERVAL} seconds...\n")
    time.sleep(INTERVAL)

Batch loop started — Interrupt kernel to stop

[Batch 1] 2026-06-19 03:50:38
  Raw rows:   4669
  Clean rows: 4669

  Saving → dbo.CleanedAirQuality
    Rows: 4669
    ✅ Done → dbo.CleanedAirQuality

  Saving → dbo.CitySummary
    Rows: 5
    ✅ Done → dbo.CitySummary

  Saving → dbo.PollutantAvg
    Rows: 5
    ✅ Done → dbo.PollutantAvg

  Saving → dbo.DangerousAirQuality
    Rows: 0
    ✅ Done → dbo.DangerousAirQuality
  ✅ Batch 1 completed successfully
  Waiting 60 seconds...


[Batch 2] 2026-06-19 03:51:45
  Raw rows:   4673
  Clean rows: 4673

  Saving → dbo.CleanedAirQuality
    Rows: 4673
    ✅ Done → dbo.CleanedAirQuality

  Saving → dbo.CitySummary
    Rows: 5
    ✅ Done → dbo.CitySummary

  Saving → dbo.PollutantAvg
    Rows: 5
    ✅ Done → dbo.PollutantAvg

  Saving → dbo.DangerousAirQuality
    Rows: 0
    ✅ Done → dbo.DangerousAirQuality
  ✅ Batch 2 completed successfully
  Waiting 60 seconds...


[Batch 3] 2026-06-19 03:52:54
